In [ ]:
import os
import torch
import pandas as pd

from tqdm import tqdm

from stock_bpt.stock_bpt import StockBPT, LinearModel, NaiveModel
from stock_bpt.dataloader_builder_bpt import build_dataloaders
from setup import StockBPT_cfg, LinearModel_cfg, NaiveModel_cfg
from setup import path_data_preprocessor
from stock_bpt.model_training_bpt import model_setup, train_model_cuda

from stock_bpt.model_training_bpt import train_model_cuda, evaluate_model, evaluate_best_model
from stock_bpt.model_analysis_bpt import test_model, print_loss_analysis, process_losses, format_num, process_result, store_result

In [2]:
cuda = True if torch.cuda.is_available() else False

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.13.0+cu132
CUDA build: 13.2
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


## MODEL TRAINING ---------------------------

In [3]:
torch.manual_seed(1234)
print(path_data_preprocessor)
dls, train_norms = build_dataloaders(path_data_preprocessor)

preprocessed_data/data_1min_2021_2026
Building DataLoaders...
Train dataset samples: 186,541
Train loader batches:  728
Batch size:            256


In [4]:
optimizer_data = [torch.optim.AdamW, 0.0004, 0.1]
scaler_data = [torch.amp.GradScaler, "cuda"]

max_epochs = 10

eval_bs = 1000

stockBPT, stockBPT_params, opt1, sca1, sch1 = model_setup(StockBPT, StockBPT_cfg, train_norms, device,
                                                *optimizer_data, *scaler_data)
linearModel, linearModel_params, opt2, sca2, sch2 = model_setup(LinearModel, LinearModel_cfg, train_norms, device, 
                                                      *optimizer_data, *scaler_data)
naiveModel = NaiveModel(NaiveModel_cfg, train_norms)
naiveModel.to(device)

Input Norm: torch.Size([12])|torch.Size([12])
Target Norm: torch.Size([4])|torch.Size([4])
3261440
5376


NaiveModel()

In [6]:
model_train_losses, model_val_losses = train_model_cuda(stockBPT, device, opt1, sca1, sch1, max_epochs, 
                                                        dls["train"], dls["val"], eval_bs)
linear_train_losses, linear_val_losses = train_model_cuda(linearModel, device, opt2, sca2, sch2, max_epochs,
                                                        dls["train"], dls["val"], eval_bs)


Continuing from previous checkpoint...
[] []


|          | 0.0% (00:00) Setting up...                                                                   

Epoch 4:

Learning Rate: 4.00e-04



|█▍        | 14.3% (11:39) Evaluating model on validation data... (223/224) [1680/11760]:                 

Epoch 4:
Training Loss:
   (MAE) 0.3742838203907013
   ($$$) 0.0028559528291225433
   (NLL) 0.13794617354869843
Validation Loss:
   (MAE) 0.3950866460800171
   ($$$) 0.003014716086909175
   (NLL) 0.2672916054725647

Best Validation: 0.08423148840665817
----------------------------------------------------------------------------------------------------

Learning Rate: 4.00e-04



...

|████████  | 80.0% (03:36) Evaluating model on validation data... (223/224) [13440/16800]: 

Epoch 8:
Training Loss:
   (MAE) 0.3670428693294525
   ($$$) 0.0028008511289954185
   (NLL) 0.05830433592200279
Validation Loss:
   (MAE) 0.38789331912994385
   ($$$) 0.0029599731788039207
   (NLL) 0.21904373168945312

Best Validation: 0.21904373168945312
----------------------------------------------------------------------------------------------------

Learning Rate: 4.00e-04



|█████████ | 90.0% (04:03) Evaluating model on validation data... (223/224) [15120/16800]: 

Epoch 9:
Training Loss:
   (MAE) 0.3665784001350403
   ($$$) 0.002797333989292383
   (NLL) 0.009098958224058151
Validation Loss:
   (MAE) 0.38742923736572266
   ($$$) 0.002956458367407322
   (NLL) 0.17747759819030762

Best Validation: 0.17747759819030762
----------------------------------------------------------------------------------------------------

Learning Rate: 4.00e-04



Epoch 10:
Training Loss:
   (MAE) 0.3668743073940277
   ($$$) 0.002799609676003456
   (NLL) 0.03771132230758667
Validation Loss:
   (MAE) 0.3877236247062683
   ($$$) 0.0029587242752313614
   (NLL) 0.2048995941877365

Best Validation: 0.17747759819030762
----------------------------------------------------------------------------------------------------

Finished


## Model Analysis -------------------------

In [7]:
#* REUSES OBJETCS FROM TRAINING
analysis_steps = min(eval_bs, len(dls["train"])) + min(eval_bs, len(dls["val"])) + min(eval_bs, len(dls["test"]))
analysis_pbar = tqdm(total=3*analysis_steps, desc=f"Evaluating the best model parameters...".ljust(80),
                bar_format="|{bar}| {percentage:3.1f}% ({elapsed}) {desc}", position=0, leave=False)

#* Reevaluates models by their best parameters on train and val dataloaders
naive_losses = evaluate_model(dls["train"], dls["val"], naiveModel, device, eval_bs, analysis_pbar)
linear_losses = evaluate_best_model(linearModel, device, opt2, sca2, sch2, dls["train"], dls["val"], eval_bs, analysis_pbar, True)
bpt_losses = evaluate_best_model(stockBPT, device, opt1, sca1, sch1, dls["train"], dls["val"], eval_bs, analysis_pbar, True) 

#* Final evaluation on unseen test dataloader
naive_test_losses = test_model(dls["test"], naiveModel, device, eval_bs, analysis_pbar)
linear_test_losses = test_model(dls["test"], linearModel, device, eval_bs, analysis_pbar)
bpt_test_losses = test_model(dls["test"], stockBPT, device, eval_bs, analysis_pbar)


|███▏      | 32.2% (00:15) Evaluating model on training data... (8/728) [961/2988]:                       

[] []


|██████▎   | 63.7% (00:30) Evaluating model on validation data... (223/224) [1904/2988]: 

[] []


|██████████| 100.0% (03:09) Evaluating model on testing data... (43/44) [2988/2988]:     

In [8]:
for key, features in [("NLL", StockBPT_cfg["target_features"]),
                      ("MAE", StockBPT_cfg["target_features"]),
                      ("RMAE", StockBPT_cfg["target_features"]),
                      ("STD", [f"{feature}_std" for feature in StockBPT_cfg["target_features"]]),
                      ("RSTD", [f"{feature}_std" for feature in StockBPT_cfg["target_features"]]),
                      ("Z^2", StockBPT_cfg["target_features"])]:
    print_loss_analysis(process_losses(bpt_losses + bpt_test_losses +
                                       linear_losses + linear_test_losses +
                                       naive_losses + naive_test_losses, key), 
                                       [stockBPT.cfg["name"], linearModel.cfg["name"], naiveModel.cfg["name"]],
                                       [format_num(stockBPT_params), format_num(linearModel_params), "0"], 
                                       features, key)


--------------------------------------------------------------------------------------------------------------

NLL

--------------------------------------------------------------------------------------------------------------

                    o        h        l        c        
StockGPT-v11-2.25-0.75-183: 3.3M
    Training:       -0.1730  -0.2132  -0.1651  -0.1606    >  -0.1780
    Validation:     -0.0101  -0.0433  -0.0000  0.0041     >  -0.0123
    Testing:        -0.1089  -0.1442  -0.0958  -0.0920    >  -0.1102
    
LinearModel-v11-2.25-0.75-183: 5.4K
    Training:       0.0423   -0.0106  0.0002   0.0044     >  0.0091
    Validation:     0.1996   0.1590   0.1707   0.1806     >  0.1775
    Testing:        0.0884   0.0493   0.0729   0.0745     >  0.0713
    
NaiveModel-B1_183: 0
    Training:       1.6797   1.6697   1.6769   1.6739     >  1.6750
    Validation:     1.6838   1.6732   1.6834   1.6788     >  1.6798
    Testing:        1.6824   1.6701   1.6819   1.6761     >  1.677

In [9]:
dls, train_norms = build_dataloaders("handpicked_data", False, drop_last = False)

Building DataLoaders...


In [10]:
test_losses = test_model(dls["test"], stockBPT, device, eval_bs)
print(test_losses)

({'NLL': tensor([1.5161, 1.4093, 1.4234, 1.6082], device='cuda:0'), 'MAE': tensor([0.9936, 0.8383, 0.8532, 0.9890], device='cuda:0'), 'RMAE': tensor([0.0075, 0.0064, 0.0065, 0.0076], device='cuda:0'), 'STD': tensor([2.4492, 2.1726, 2.1553, 1.9352], device='cuda:0'), 'RSTD': tensor([0.0186, 0.0167, 0.0164, 0.0148], device='cuda:0'), 'Z^2': tensor([0.2910, 0.3910, 0.4376, 0.5785], device='cuda:0')},)
